In [43]:
import copy
import numpy as np

class treeNode(object):
    def __init__(self, pcd, parent=None, depth=0):
        self.pcd = pcd  # str
        self.y = '' if depth !=0 else 'root' # str
        self.parent = parent  # treeNode
        self.numVisits = 0  # int
        self.V = 0  # float
        self.children = {}  # dict{str:treeNode}
        self.depth = depth  # int
        self.isFullyExpanded = False  # expanded
        self.visit_sequence = 0
        self.final_ans_flag = 0
        self.isTerminal = False  # value acceptable
        self.on_final_route = False
        self.min_steps_to_correct = 1024
        self.summary = ''
        self.he = 0  # hard estimation
        self.se = 0  # soft estimation

    def __str__(self):
        s = ["numVisits: %d" % self.numVisits, 
             f'V:{self.V}', "possibleActions: %s" % (self.children.keys()), 
             f'he:{self.he}', f'se:{self.se}', 
             f'isFullyExpanded:{self.isFullyExpanded}', 
             f'visit_sequence:{self.visit_sequence}', 
             f'final_ans_flag:{self.final_ans_flag}', 
             f'isTerminal:{self.isTerminal}', 
             f'on_final_route:{self.on_final_route}',
             f'min_steps_to_correct:{self.min_steps_to_correct}', 
             f'summary:{self.summary}']
        return "%s: {%s}" % (self.__class__.__name__, ', '.join(s))

    def append_children(self, new_pcd: str):
        node = treeNode(new_pcd, self, self.depth + 1)
        node.update_y_from_parent()
        self.children.update({new_pcd: node})
        return node

    def update_y_from_parent(self):
        if self.parent is None:
            self.y = self.pcd
        else:
            self.y = self.parent.y + '->' + self.pcd

    def update_value(self, value):
        self.V = value


    def getBestV(self):  # Gets the subtree maximum value node
        if not self.isFullyExpanded: # watch out the meaning of isFullyExpanded
            return self, self.V
        max_V = self.V
        max_node = self
        for child in self.children.values():
            subNode, subValue = child.getBestV()
            if subValue >= max_V:
                max_V = subValue
                max_node = subNode
        return max_node, max_V

    def trace_route(self):  # trace route from terminal node to root
        cur_node = self
        while cur_node is not None:
            cur_node.on_final_route = True
            cur_node = cur_node.parent

    def get_new_value_samples(self):  # get value samples from search tree (start from terminal node)
        if self.depth == 0:
            return []
        step_value = 1.0 / self.depth
        new_samples = []
        cur_node = self.parent
        while cur_node is not None:
            for child in cur_node.children.values():
                if child.on_final_route:
                    child_value = step_value * child.depth
                    new_item = {'steps': child.y, 'value': child_value}
                    new_samples.append(new_item)
                else:
                    child_value = max(step_value * (cur_node.depth - 1), 0)
                    new_item = {'steps': child.y, 'value': child_value}
                    new_samples.append(new_item)
            cur_node = cur_node.parent
        return new_samples


    def get_all_end_root_nodes_prm(self):
        end_nodes = []
        if self.isFullyExpanded:
            for child in self.children.values():
                end_nodes.extend(child.get_all_end_root_nodes_prm())
            return end_nodes
        else:
            if self.isTerminal:
                return [self]
            else:
                return []

    def get_all_value_samples_vm(self):
        full_value_samples = []
        if self.depth == 0:
            self.V = 0
        else:
            if self.he == 0:
                r = -1
            else:
                r = 1
            self.V = max(0, (1 - self.parent.V) * r / self.min_steps_to_correct + self.parent.V)
            full_value_samples.append({'steps': self.y, 'value': self.V})
        if self.isFullyExpanded:
            for child in self.children.values():
                if child.min_steps_to_correct < 1024:
                    sub_samples = child.get_all_value_samples_vm()
                    full_value_samples.extend(sub_samples)
        return full_value_samples

    def get_full_value_samples_vm(self, end_leaf_nodes):
        for leaf in end_leaf_nodes:
            if leaf.min_steps_to_correct > 1:
                continue
            else:
                leaf.he = 1
                cur_node = leaf.parent
                while cur_node is not None:
                    cur_node.min_steps_to_correct = min(
                        [n.min_steps_to_correct for n in cur_node.children.values()]) + 1
                    cur_node.he = 1
                    cur_node = cur_node.parent
        for leaf in end_leaf_nodes:
            if leaf.min_steps_to_correct > 1:
                cur_node = leaf.parent
                while cur_node is not None and cur_node.min_steps_to_correct == 1024:
                    cur_node = cur_node.parent
                if cur_node is None:
                    continue
                else:
                    m = cur_node.min_steps_to_correct
                    cur_node = leaf
                    while cur_node.min_steps_to_correct == 1024:
                        cur_node.min_steps_to_correct = m
                        cur_node = cur_node.parent
            else:
                continue
        value_samples = self.get_all_value_samples_vm()
        return value_samples

    def get_all_value_samples_prm(self):
        full_value_samples = []
        if self.on_final_route:
            full_value_samples.append({'steps': self.y, 'value': self.he})
            if self.isFullyExpanded:
                for child in self.children.values():
                    if child.on_final_route:
                        sub_samples = child.get_all_value_samples_prm()
                        full_value_samples.extend(sub_samples)
            return full_value_samples
        else:
            return []

    def get_full_value_samples_prm(self, end_leaf_nodes):
        for leaf in end_leaf_nodes:
            cur_node = leaf.parent
            while cur_node is not None:
                cur_node.on_final_route = True
                cur_node = cur_node.parent
        for leaf in end_leaf_nodes:
            cur_node = leaf.parent
            while cur_node is not None:
                cur_node.he = max([n.he for n in cur_node.children.values() if n.on_final_route])
                cur_node = cur_node.parent
        value_samples = self.get_all_value_samples_prm()
        return value_samples

In [44]:
# 测试1: 基础树结构构建
root = treeNode("Root", depth=0)
child1 = root.append_children("A")
child2 = root.append_children("B")
root.isFullyExpanded = True
grandchild = child1.append_children("A1")

print("=== 测试1: 树结构验证 ===")
print(f"根节点子节点数: {len(root.children)} (预期: 2)")
print(f"子节点A的深度: {child1.depth} (预期: 1)")
print(f"孙子节点路径: {grandchild.y} (预期: RootA1)")


=== 测试1: 树结构验证 ===
根节点子节点数: 2 (预期: 2)
子节点A的深度: 1 (预期: 1)
孙子节点路径: root->A->A1 (预期: RootA1)


In [45]:
print(root)

treeNode: {numVisits: 0, V:0, possibleActions: dict_keys(['A', 'B']), he:0, se:0, isFullyExpanded:True, visit_sequence:0, final_ans_flag:0, isTerminal:False, on_final_route:False, min_steps_to_correct:1024, summary:}


In [46]:

# 测试2: 价值样本收集
grandchild.on_final_route = True
grandchild.depth = 2
samples = grandchild.get_new_value_samples()

print("\n=== 测试2: 价值样本生成 ===")
print(f"收集到样本数: {len(samples)} (预期: 3)")
for sample in samples:
    print(f"路径: {sample['steps']} | 价值: {sample['value']:.2f}")



=== 测试2: 价值样本生成 ===
收集到样本数: 3 (预期: 3)
路径: root->A->A1 | 价值: 1.00
路径: root->A | 价值: 0.00
路径: root->B | 价值: 0.00


In [47]:

# 测试3: 终端节点检测
terminal_node = treeNode("END", parent=child2)
terminal_node.isTerminal = True
terminal_node.V = 0.9
end_nodes = root.get_all_end_root_nodes_vm(end_gate=0.8)

print("\n=== 测试3: 终端节点检测 ===")
print(f"找到终端节点: {len(end_nodes)} (预期: 1)")
print(f"终端节点路径: {end_nodes[0].y}")


cuttent node: root
cuttent node: root->A
cuttent node: root->B

=== 测试3: 终端节点检测 ===
找到终端节点: 0 (预期: 1)


IndexError: list index out of range

In [48]:
print(child2)

treeNode: {numVisits: 0, V:0, possibleActions: dict_keys([]), he:0, se:0, isFullyExpanded:False, visit_sequence:0, final_ans_flag:0, isTerminal:False, on_final_route:False, min_steps_to_correct:1024, summary:}


In [ ]:

# 测试4: 价值计算验证
root.V = 0.5
child1.he = 1
child1.min_steps_to_correct = 2
value_samples = child1.get_all_value_samples_vm()

print("\n=== 测试4: 价值计算 ===")
print(f"计算后的V值: {child1.V:.2f} (预期: 0.75)")
print(f"收集到价值样本: {len(value_samples)} (预期: 1)")

# 测试5: 完整价值传播
leaf = grandchild.append_children("A1a")
leaf.min_steps_to_correct = 1
full_samples = root.get_full_value_samples_vm([leaf])

print("\n=== 测试5: 完整价值传播 ===")
print(f"传播后根节点V值: {root.V:.2f} (预期: 0.00)")
print(f"总样本数量: {len(full_samples)} (预期: 3)")